# Scraping: récupération de la page d'accueil de la RTBF

Dans ce notebook, nous créons un robot qui va ouvrir la page d'accueil de la [RTBF](https://www.rtbf.be/) et récupérer le titre de tous les articles du jour et les stocker dans un fichier csv.

## Imports

In [1]:
import re
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

## Récupération de tous les articles de la page d'accueil



In [6]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}

articles = []

root_url = f"https://www.rtbf.be"
response = requests.get(root_url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
for link in soup.find_all("a", {"class":"stretched-link outline-hidden"}):
    title = link.text.strip()
    url = link.get('href') if not link.get('href').startswith('http') else link.get('href')
    articles.append([url, title])

In [7]:
# Affichage du nombre d'articles récupérés
len(articles)

92

In [8]:
# Affichage des 10 premières entrées
articles[:10]

[['https://www.rtbf.be/short/lastuce-pour-etre-bon-aux-jeux-de-plateforme-6409?flux=main-1',
  ''],
 ['https://www.rtbf.be/short/six-semaines-apres-son-accouchement-barbara-palvin-defile-pour-victorias-secret-entre-critiques-et-glorification-6407?flux=main-1',
  ''],
 ['https://www.rtbf.be/short/france-le-sud-est-du-pays-touche-par-de-fortes-intemperies-6401?flux=main-1',
  ''],
 ['https://www.rtbf.be/short/israel-des-guerres-et-des-fractures-6404?flux=main-1',
  ''],
 ['https://www.rtbf.be/short/envdv-20261021-reel-melanie-mouches-6402?flux=main-1',
  ''],
 ['https://www.rtbf.be/short/la-foire-de-liege-il-y-a-60-ans-6398?flux=main-1',
  ''],
 ['https://www.rtbf.be/short/crash-test-final-fantasy-resonance-6396?flux=main-1',
  ''],
 ['https://www.rtbf.be/short/dou-vient-le-slogan-acab-6395?flux=main-1', ''],
 ['https://www.rtbf.be/short/envdv-nico-apu-6399?flux=main-1', ''],
 ['https://www.rtbf.be/short/portugal-des-campements-pour-denoncer-la-crise-du-logement-6391?flux=main-1',
  '']]

## Création d'un dataframe avec les liens et les titres des articles


In [5]:
df = pd.DataFrame(articles, columns=['link', 'title'])
df

,link,title
0,https://www.rtbf.be/short/lastuce-pour-etre-bo...,
1,https://www.rtbf.be/short/six-semaines-apres-s...,
2,https://www.rtbf.be/short/france-le-sud-est-du...,
3,https://www.rtbf.be/short/israel-des-guerres-e...,
4,https://www.rtbf.be/short/envdv-20261021-reel-...,
...,...,...
87,https://auvio.rtbf.be/media/le-8-9-le-8-9-3541102,Le 8-9
88,https://auvio.rtbf.be/media/un-si-grand-soleil...,Un si grand soleil
89,https://auvio.rtbf.be/media/le-6-8-le-6-8-3541056,Le 6-8
90,https://auvio.rtbf.be/media/les-feux-de-l-amou...,Feux de l'amour S27


In [9]:
# Sauvegarde du dataframe dans un fichier csv
df.to_csv(f"../data/rtbf_{time.strftime('%Y%m%d')}.csv", index=False)

## Téléchargement d'un article et affichage du texte

In [10]:
# Récupération du premier lien du dataframe
article_url = df.iloc[36,0]

# Ouverture de la page
response = requests.get(article_url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')
# Parsing du contenu
title = soup.find("h1").get_text(separator=' ')

paragraphs = []
for paragraph in soup.find_all("p", attrs={"class": None}):
    paragraphs.append(paragraph.get_text(separator=' ').strip())

content = "\n".join(paragraphs)
print(title)
print("==================================")
print(content)

Champions League
C’était presque une finale avant l’heure, un match d’automne alors que la grandeur attend en général le printemps. Et le tenant du titre n’a eu aucune pitié pour l’équipe pourtant insolente de talent qu’est le FC Barcelone, leader invaincu de la Liga : le PSG a gagné en affolant les chiffres.
4-2, déjà : le score final est révélateur. Paris encaisse, oui. Mais Paris enfile aussi les buts en sa faveur, même sans ses attaquants titulaires (le PSG a fini le match et marqué ses deux derniers buts sans  Ousmane Dembélé  ni  Khvicha Kvaratskhelia ,  sortis sur blessure , et sans  Désiré Doué , premier buteur du match, remplacé à la 78e).
Pour marquer, Paris applique une recette assez simple : Paris tente. Beaucoup. Face à une équipe joueuse comme le Barça et dans une phase de la compétition où les équipes peuvent se le permettre, le jeu offensif et léché des Parisiens fait des miracles. Ce mardi, le PSG a cadré près de 70% de ses tirs et n’est pas passé loin de son record en

### Nettoyage du texte à l'aide d'expressions régulières

In [11]:
# Suppression de tout ce qui se trouve entre parenthèses
clean_content = re.sub("\([^\)]+\)", " ", content)
# Suppression des espaces multiples
clean_content = re.sub("\s+", " ", clean_content)

print(content)
print("==================================")
print(clean_content)


C’était presque une finale avant l’heure, un match d’automne alors que la grandeur attend en général le printemps. Et le tenant du titre n’a eu aucune pitié pour l’équipe pourtant insolente de talent qu’est le FC Barcelone, leader invaincu de la Liga : le PSG a gagné en affolant les chiffres.
4-2, déjà : le score final est révélateur. Paris encaisse, oui. Mais Paris enfile aussi les buts en sa faveur, même sans ses attaquants titulaires (le PSG a fini le match et marqué ses deux derniers buts sans  Ousmane Dembélé  ni  Khvicha Kvaratskhelia ,  sortis sur blessure , et sans  Désiré Doué , premier buteur du match, remplacé à la 78e).
Pour marquer, Paris applique une recette assez simple : Paris tente. Beaucoup. Face à une équipe joueuse comme le Barça et dans une phase de la compétition où les équipes peuvent se le permettre, le jeu offensif et léché des Parisiens fait des miracles. Ce mardi, le PSG a cadré près de 70% de ses tirs et n’est pas passé loin de son record en la matière dans 

<>:2: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\('
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_54533/3799268790.py:2: SyntaxWarning: invalid escape sequence '\('
  clean_content = re.sub("\([^\)]+\)", " ", content)
/tmp/ipykernel_54533/3799268790.py:4: SyntaxWarning: invalid escape sequence '\s'
  clean_content = re.sub("\s+", " ", clean_content)


### Création d'un fichier avec le contenu de l'article


In [12]:
with open("../data/rtbf_example.txt", "w") as writer:
    writer.write(f"{title}\n\n{clean_content}")

## Pour en savoir plus

- Le web scraping avec Python: https://realpython.com/beautiful-soup-web-scraper-python/
- Tutoriel sur les expressions régulières: https://www.w3schools.com/python/python_regex.asp